# Load Packages

In [1]:
using DifferentialEquations
using OrdinaryDiffEq
using DiffEqBase
using Sundials
using ODEInterfaceDiffEq
using Plots
using Measures
using CSV
using DataFrames
using EasyFit
using StatsPlots
using LinearAlgebra
using Random
using Distributions
using LsqFit
using BlackBoxOptim
using LaTeXStrings
using JLD2
using BlackBoxOptim: num_func_evals
using LatinHypercubeSampling
using Gtk
using XLSX

include("ModelFunctionsAll.jl")

findTTP (generic function with 1 method)

# Load Data and Pyruvate T1s

In [ ]:
# With this block it will open a Julia window where we need to select the "temporaldataALL.csv file generated by the Python code.
# this file will be located in the experimental folder, and then in the folder with the 13Cdata. Make sure you are in the corresponding folder not the one from the previous analysis.
# Once selected this file, the kinetic data will be loaded and can proceed with the next steps.
using Gtk
filename = open_dialog("Load Batch Data File for Fitting! ;) ")

barr = findall("\\", filename)[end][1];
pathD = filename[1:barr];
filNam = filename[barr+1:end];

xf = XLSX.readxlsx(pathD*filNam);
shn = XLSX.sheetnames(xf);
df = DataFrame(XLSX.readtable(pathD*filNam, shn[1]));
df

### Extract Data

In [ ]:
PathsAll = df[1:end, 2]
global DAT = Array{Any}(undef, length(PathsAll))
global startScanAll = Array{Any}(undef, length(PathsAll))
global StartInjectAll = Array{Any}(undef, length(PathsAll))
global delaAll = Array{Any}(undef, length(PathsAll))
Delay_auto = Array{Any}(undef, length(PathsAll))
# Load and plot experimental data generated by the DataProcGEN_Magnitude SCRIPT
for i in 1:length(PathsAll)
    DAT[i] = Matrix(CSV.read(PathsAll[i]*"TemporalDataALL.csv", DataFrame));
    startScanAll[i] = df[i, 3]
    StartInjectAll[i] = df[i, 4]

    Delay_auto[i] = findfirst(diff(DAT[i][:,2]) .> 1e6) -1
    try 
        delaAll[i] = convert(Int, round((StartInjectAll[i]-startScanAll[i])/5)) #+ 1;
    catch
        delaAll[i] = findfirst(diff(DAT[i][:,2]) .> 1e6) -1
    end
end

for i in 1:length(DAT)
    DAT[i][:,2] = DAT[i][:,2].-mean(DAT[i][:,2][end-10:end])
    DAT[i][:,3] = DAT[i][:,3].-mean(DAT[i][:,3][end-10:end])
    DAT[i][:,4] = DAT[i][:,4].-mean(DAT[i][:,4][end-10:end])

    ss1 = scatter(DAT[i][:,1]./60, DAT[i][:,2], label = "", grid = false, xlabel = "time (min)", ylabel = "Pyruvate (A.U.)")
    ss2 = scatter(DAT[i][:,1]./60, DAT[i][:,3], label = "", grid = false, xlabel = "time (min)", ylabel = "Alanine (A.U.)", color = "red", title = df[i, 1])
    ss3 = scatter(DAT[i][:,1]./60, DAT[i][:,4], label = "", grid = false, xlabel = "time (min)", ylabel = "Lactate (A.U.)", color = "green")

    display(plot(ss1, ss2, ss3, layout = (1,3), size = (1400, 400), margin = 10mm))
end

# T1 Approximations

In [ ]:
T1sAll = Array{Any}(undef, length(PathsAll))
for i in 1:length(PathsAll)
    T1sAll[i] = T1Approx(DAT[i]);
end

# Parameter Estimation

In [ ]:
ppAll = Array{Any}(undef, length(PathsAll))


for i in 1:length(PathsAll)
    global dela = delaAll[i]
    global dat2 = DAT[i]
    global T1s = T1sAll[i]
    # Testing and initialising function. Does nothing for you, but you need to run it. 
    ObjectFunctME([1,1,1,1,1,1,1,100000])

    #This part of the code will try to fit a matematical model using iterations. It may take 2 minutes to finish the fitting.
    # wait until it finishes and appears the graph and the parameter matrix before running the next block

    ppAll[i] = EstimateParsMod(pathD, filNam, T1s, true)

end


#### Parameters Summary

In [ ]:
for i in 1:length(PathsAll)
    println("----- "*df[i, 1]*": ")
    println("kin_B = "*string(ppAll[i][1]))
    println("kin_O = "*string(ppAll[i][2]))
    println("kpl = "*string(ppAll[i][3]))
    println("kal = "*string(ppAll[i][4]))
    println("T1_P = "*string(ppAll[i][5]))
    println("T1_L = "*string(ppAll[i][6]))
    println("T1_A = "*string(ppAll[i][7]))

    println("PyrO_B__0 = "*string(ppAll[i][8]))
    println("############################ \n")
end


# nms = ["kin_B","kin_O","kpl","kal","T1_P","T1_L","T1_A","PyrO_B__0"]
# CSV.write(pathD*"ParameterValues.csv", DataFrame(hcat(nms, pp), :auto))

#### Plot Results

In [ ]:
#Check that the graph are well adjusted to the points, and that the first point of the line is a zero in the graph
# If the first point of the line is not a zero in the pyruvate graph, adjust the startInject time
SSEall, objAll = zeros(length(PathsAll)), zeros(length(PathsAll))

for i in 1:length(PathsAll)
    SSEall[i], objAll[i] = plotFit(DAT[i], delaAll[i], ppAll[i], PathsAll[i], "temporaldataALL.csv");
end

### Find time of maximum peak (from mathematical equations)

In [ ]:
ttpPAll, ttpAAll, ttpLAll = Array{Any}(undef, length(PathsAll)), Array{Any}(undef, length(PathsAll)), Array{Any}(undef, length(PathsAll))

for i in 1:length(PathsAll)
    println("Experiment "*string(i))
    ttpPAll[i], ttpAAll[i], ttpLAll[i] = findTTP(DAT[i], delaAll[i], ppAll[i], PathsAll[i], "temporaldataALL.csv")
end

# Save All Data in Same File

In [ ]:
caps = ["ID", "FilePath", "StartScan", "StartInject", "Delay", "Delay_auto", "kin_B", "kin_O", "kpl", "kal", "T1_P", "T1_L", "T1_A","Pyr0", "TTP_Pyr", "TTP_Ala", "TTP_Lac", "SSE", "Obj"]
FinDat = Array{Any}(undef, length(PathsAll), length(caps))
FinDat2 = Array{Any}(undef, length(PathsAll)+1, length(caps))

FinDat2[1,:] = caps
for i in 1:length(PathsAll)
    FinDat[i,1] = df[i, 1]
    FinDat[i,2] = df[i, 2]
    FinDat[i,3] = startScanAll[i]
    FinDat[i,4] = StartInjectAll[i]
    FinDat[i,5] = delaAll[i]
    FinDat[i,6] = Delay_auto[i]
    FinDat[i,7:14] = ppAll[i]
    FinDat[i,15] = ttpPAll[i]
    FinDat[i,16] = ttpAAll[i]
    FinDat[i,17] = ttpLAll[i]
    FinDat[i,18] = SSEall[i]
    FinDat[i,19] = objAll[i]

    FinDat2[i+1,1] = df[i, 1]
    FinDat2[i+1,2] = df[i, 2]
    FinDat2[i+1,3] = startScanAll[i]
    FinDat2[i+1,4] = StartInjectAll[i]
    FinDat2[i+1,5] = delaAll[i]
    FinDat2[i+1,6] = Delay_auto[i]
    FinDat2[i+1,7:14] = ppAll[i]
    FinDat2[i+1,15] = ttpPAll[i]
    FinDat2[i+1,16] = ttpAAll[i]
    FinDat2[i+1,17] = ttpLAll[i]

    FinDat2[i+1,18] = SSEall[i]
    FinDat2[i+1,19] = objAll[i]
end

df2 = DataFrame(FinDat, :auto)
rename!(df2, caps)

df3 = DataFrame(FinDat2, :auto)

CSV.write(pathD*"\\KineticResults_"*filNam[1:end-5]*".csv", df2);

XLSX.openxlsx(pathD*"\\KineticResults_"*filNam[1:end-5]*".xlsx",mode="w") do xf
     sheet = xf["Sheet1"]
     for r in 1:size(df3,1), c in 1:size(df3,2)
          sheet[XLSX.CellRef(r , c )] = df3[r,c]
     end
end

df2

#                       THE END 